In [1]:
import qsample as qs
import numpy as np
import matplotlib.pyplot as plt

eft = qs.Circuit(noisy=True)
sz_123 = qs.Circuit(noisy=True)
meas7 = qs.Circuit(noisy=False)

eft.from_stim_circuit("""R 0 1 2 3 4 5 6 7
                        H 0 1 3
                        CNOT 0 4
                        CNOT 1 2
                        TICK
                        CNOT 3 5
                        TICK
                        CNOT 0 6
                        TICK
                        CNOT 3 4
                        TICK
                        CNOT 1 5
                        TICK
                        CNOT 0 2
                        TICK
                        CNOT 5 6
                        TICK
                        CNOT 4 7
                        TICK
                        CNOT 2 7
                        TICK
                        CNOT 5 7
                        M 7""")

sz_123.from_stim_circuit("""R 8
CNOT 0 8
                        TICK
                        CNOT 1 8
                        TICK
                        CNOT 3 8
                        TICK
                        CNOT 6 8
                            M 8""")

meas7.from_stim_circuit("""M 0 1 2 3 4 5 6""")





k1 = 0b0001111
k2 = 0b1010101
k3 = 0b0110011
k12 = k1 ^ k2
k23 = k2 ^ k3
k13 = k1 ^ k3
k123 = k12 ^ k3
stabilizerGenerators = [k1, k2, k3]
stabilizerSet = [0, k1, k2, k3, k12, k23, k13, k123]

def hamming2(x, y):
    count, z = 0, x ^ y
    while z:
        count += 1
        z &= z - 1
    return count

fails = []
def logErr(out):
    global fails
    c = np.array([hamming2(out, i) for i in stabilizerSet])
    d = np.flatnonzero(c <= 1)
    e = np.array([hamming2(out ^ (0b1111111), i) for i in stabilizerSet])
    f = np.flatnonzero(e <= 1)
    if len(d) != 0:
        return False
    elif len(f) != 0:
        fails.append(out)
        return True
    if len(d) != 0 and len(f) != 0: 
        raise('-!-!-CANNOT BE TRUE-!-!-')

def flagged_z_look_up_table_1(z):
    s = [z]

    if s == [1]:
        return True
    else: 
        return False

functions = {"logErr": logErr, "lut": flagged_z_look_up_table_1}

steane0 = qs.Protocol(check_functions=functions, fault_tolerant=True)

steane0.add_nodes_from(['ENC', 'Z2', 'meas'], circuits=[eft, sz_123, meas7])
steane0.add_node('X_COR', circuit=qs.Circuit(noisy=True).from_stim_circuit("""X 6"""))
steane0.add_edge('START', 'ENC', check='True')
steane0.add_edge('ENC', 'meas', check='ENC[-1]==0')
steane0.add_edge('ENC', 'Z2', check='ENC[-1]==1')
steane0.add_edge('Z2', 'X_COR', check='lut(Z2[-1])')
steane0.add_edge('Z2', 'meas', check='not lut(Z2[-1])')
steane0.add_edge('X_COR', 'meas', check='True')
steane0.add_edge('meas', 'FAIL', check='logErr(meas[-1])')

err_model = qs.noise.E1

n_err = 2
stim_sam = qs.FT_Check(protocol=steane0, simulator=qs.StimSimulator,  p_max={'q': 0.1}, w_max=n_err, err_model=err_model, L=3)
stim_sam.run(2000)

/Users/jhfontes/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
p=('1.00e-01',):   0%|          | 0/2000 [00:00<?, ?it/s]


False

In [ ]:
n_err = 1
stim_sam = qs.FT_Check(protocol=steane0, simulator=qs.StimSimulator,  
                       p_max={'q': 0.1}, w_max=n_err, err_model=err_model, L=3)
stim_sam.run(2000)

p=('1.00e-01',): 100%|██████████| 2000/2000 [00:00<00:00, 5033.29it/s]


True

In [ ]:
ghz = qs.Circuit([ {"init": {0,1,2,3,4}},
                   {"H": {0}},
                   {"CNOT": {(0,1)}},
                   {"CNOT": {(1,2)}},
                   {"CNOT": {(2,3)}},
                   {"CNOT": {(3,4)}},
                   {"CNOT": {(0,4)}},
                   {"measure": {4}}   ])

# Define protocol for 1 round of repetition

def logErr(msmt_list):
    return msmt_list[-1] == 1 # If True transition to FAIL

functions = {'logErr': logErr}

ghz1 = qs.Protocol(check_functions=functions, fault_tolerant=False)

ghz1.add_node('ghz', circuit=ghz) # Add node with corresponding circuit
ghz1.add_edge('START', 'ghz', check='True') # Transition START -> first circuit node always True
ghz1.add_edge('ghz', 'FAIL', check='logErr(ghz)')

err_model = qs.noise.E1

n_err = 1
stim_sam = qs.FT_Check(protocol=ghz1, simulator=qs.StimSimulator,  
                       p_max={'q': 0.1}, w_max=n_err, err_model=err_model, L=3)
stim_sam.run(2000)

p=('1.00e-01',):   0%|          | 0/2000 [00:00<?, ?it/s]


False